In [1]:
import torch
from embeddings.clap import CLAPWrapper
from llms.llmclient import LLMClient
from experimentation.experiment import Method, InstructionSet1, run_experiments

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

ModuleNotFoundError: No module named 'src'

In [ ]:
clap = CLAPWrapper(device=device)
llmclient = LLMClient()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

In [ ]:
context = "piano music"

instructionset_initialization = InstructionSet1(target="bright", context=context)
# → "This is piano music. Make this sound more bright."

instructionset_refinement = InstructionSet1(anchor="sharp", target="warm", context=context)
# → "This is piano music, but the sound is sharp. Make this sound more warm."

In [ ]:
results, experiment_dir = run_experiments(
    methods = [Method.InstructFX2FX, Method.LLM_LLM],
    instructionset_initialization=instructionset_initialization,
    instructionset_refinement=instructionset_refinement,
    raw_audio_paths=["../data/audio/piano.wav"],
    sample_rate=44100,
    llm_client=llmclient,
    embedding=clap,
    iterations=12,
    results_dir="../results",
    nr_of_experiments_per_file=1,
    effects=["eq", "reverb"],
    snapshot_interval=5,
)
print(f"experiment_dir: {experiment_dir}")